In [1]:
# Import libraries

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import shap 
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset

data = pd.read_parquet("NF-BoT-IoT-V2.parquet")

print("Dataset shape:", data.shape)
print("Dataset columns:", data.columns.tolist())

# naming the label columns
label_column = "Attack"                     
drop_columns = ["Attack", "Label"]           

Dataset shape: (30420086, 43)
Dataset columns: ['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label', 'Attack']


In [3]:
# Data preprocessing

print("Data Preprocessing")

# remove duplicates
before = len(data)
data.drop_duplicates(inplace=True)
after = len(data)
print(f"Removed {before - after} duplicate rows.")

Data Preprocessing
Removed 0 duplicate rows.


In [4]:
# Drop the irrelevant identifier columns (avoids learning bias)
cols_to_drop = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT"]
cols_to_drop = [col for col in cols_to_drop if col in data.columns]
data.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped identifier columns: {cols_to_drop}")

Dropped identifier columns: ['L4_SRC_PORT', 'L4_DST_PORT']


In [5]:
# Check class distribution before balancing
print(f"\n Class distribution before balancing:\n{data[label_column].value_counts()}")


 Class distribution before balancing:
Attack
DDoS              14280259
DoS               13645057
Reconnaissance     2363017
Benign              129437
Theft                 2316
Name: count, dtype: int64


In [10]:
# sampling and balancing the dataset 

MAX_SAMPLES_PER_CLASS = 50_000

df_balanced = pd.concat([
    group.sample(n=min(len(group), MAX_SAMPLES_PER_CLASS), random_state=42)
    for name, group in data.groupby(label_column)
]).reset_index(drop=True)

print(f"\nClass distribution after balancing:\n{df_balanced[label_column].value_counts()}")
print(f"Balanced dataset shape: {df_balanced.shape}")


Class distribution after balancing:
Attack
Benign            50000
DDoS              50000
DoS               50000
Reconnaissance    50000
Theft              2316
Name: count, dtype: int64
Balanced dataset shape: (202316, 41)
